# 🚄 TARDIS — Entraînement des modèles

**Objectif :** Prédire le retard moyen à l'arrivée (en minutes) à partir des caractéristiques d'un trajet.

**Pour lancer :** `Kernel → Restart & Run All`

---

| Étape | Description |
|-------|-------------|
| 1 | Imports & configuration |
| 2 | Chargement & exploration des données |
| 3 | Entraînement de tous les modèles |
| 4 | Comparaison des modèles |
| 5 | Analyse du meilleur modèle |
| 6 | Vérification finale |

## 1. Imports & configuration

In [ ]:
import sys, os

# Le notebook est à la racine — scripts/ est accessible directement
if "." not in sys.path:
    sys.path.insert(0, ".")

import warnings
import joblib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
plt.style.use("seaborn-v0_8-whitegrid")

print(f"✓ Répertoire : {os.getcwd()}")

## 2. Chargement & exploration des données

In [ ]:
from scripts.model.config import ALL_FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGET, RANDOM_STATE
from scripts.model.preprocessing import build_preprocessor, build_route_stats, get_numpy_arrays

df = pd.read_csv("cleaned_dataset.csv", parse_dates=["Date"])
df["day_of_week"] = df["Date"].dt.dayofweek

print(f"Dataset : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"Période  : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"Gares    : {df['Departure station'].nunique()} départ | {df['Arrival station'].nunique()} arrivée")
print(f"Cible    : {TARGET}")
print(f"  → Moyenne : {df[TARGET].mean():.2f} min | Écart-type : {df[TARGET].std():.2f} | Max : {df[TARGET].max():.1f}")
print(f"  → Valeurs manquantes sur la cible : {df[TARGET].isna().sum()}")

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df[TARGET].dropna(), bins=60, color="#4f46e5", alpha=0.8, edgecolor="none")
axes[0].axvline(df[TARGET].mean(), color="#ef4444", linestyle="--", label=f"Moyenne : {df[TARGET].mean():.1f} min")
axes[0].set_xlabel("Retard (min)")
axes[0].set_ylabel("Fréquence")
axes[0].set_title("Distribution des retards à l'arrivée")
axes[0].legend()

monthly = df.groupby(["year", "month"])[TARGET].mean().reset_index()
monthly["period"] = pd.to_datetime(monthly[["year", "month"]].assign(day=1))
axes[1].plot(monthly["period"], monthly[TARGET], color="#4f46e5", linewidth=1.5)
axes[1].axhline(df[TARGET].mean(), color="#ef4444", linestyle="--", alpha=0.7)
axes[1].set_xlabel("")
axes[1].set_ylabel("Retard moyen (min)")
axes[1].set_title("Évolution mensuelle")

plt.tight_layout()
plt.show()

## 3. Préparation des données & split

**Stratégie anti-fuite de données :**  
On exclut toutes les colonnes qui encodent directement le retard à l'arrivée (comptes de trains en retard, taux de ponctualité, etc.).
Seules les features connues *avant* ou *indépendamment* du retard sont utilisées.

In [ ]:
df_model = df.dropna(subset=[TARGET]).copy()
X = df_model[ALL_FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

preprocessor = build_preprocessor()
X_train_np, X_test_np, y_train_np, y_test_np = get_numpy_arrays(
    preprocessor, X_train, X_test, y_train, y_test
)

print(f"Échantillon d'entraînement : {len(X_train):,} lignes")
print(f"Échantillon de test        : {len(X_test):,} lignes")
print(f"Dimensions après encodage  : {X_train_np.shape[1]} features")
print(f"\nFeatures numériques  : {NUMERIC_FEATURES}")
print(f"Features catégor.    : {CATEGORICAL_FEATURES}")

In [ ]:
# Baseline : prédire la moyenne (seuil minimal à battre)
from scripts.model.evaluation import baseline_metrics

baseline = baseline_metrics(y_train_np, y_test_np)
print("Baseline (prédire la moyenne d'entraînement) :")
print(f"  RMSE : {baseline['RMSE']:.3f} min")
print(f"  MAE  : {baseline['MAE']:.3f} min")
print(f"  R²   : {baseline['R2']:.4f}")

## 4. Entraînement de tous les modèles

| Modèle | Type | Avantages |
|--------|------|-----------|
| Régression linéaire | Linéaire | Interprétable, rapide |
| Ridge | Linéaire régularisé | Réduit le surapprentissage |
| Random Forest | Ensemble (bagging) | Robuste, gère les non-linéarités |
| Gradient Boosting | Ensemble (boosting) | Précis sur données tabulaires |
| XGBoost | Boosting optimisé | Très performant, régularisé |
| LightGBM | Boosting feuille | Rapide, souvent meilleur sur grandes données |

In [ ]:
from scripts.model.train import run_training

# Mettre à True pour inclure PyTorch / TensorFlow (peut crasher le kernel)
ENABLE_TORCH = False
ENABLE_TF    = False

best_pipeline, all_results, results_df = run_training(
    dataset_path="cleaned_dataset.csv",
    model_out="model.joblib",
    route_stats_out="route_stats.csv",
    models_dir="models",
    enable_torch=ENABLE_TORCH,
    enable_tf=ENABLE_TF,
)

## 5. Comparaison des modèles

In [ ]:
# Tableau récapitulatif
all_data = results_df[["RMSE", "MAE", "R2"]].copy()
all_data.loc["Baseline"] = [baseline["RMSE"], baseline["MAE"], baseline["R2"]]

print(f"{'Modèle':35s} {'RMSE':>8} {'MAE':>8} {'R²':>8} {'vs Baseline':>12}")
print("-" * 75)
for name, row in all_data.sort_values("RMSE").iterrows():
    gain = (1 - row["RMSE"] / baseline["RMSE"]) * 100
    marker = " ◄ MEILLEUR" if name == all_data.sort_values("RMSE").index[0] else ""
    bline  = " (référence)" if name == "Baseline" else f"{gain:+.1f} %"
    print(f"{name:35s} {row['RMSE']:8.3f} {row['MAE']:8.3f} {row['R2']:8.4f} {bline:>12}{marker}")

In [ ]:
# Visualisation comparative
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
colors_map = {"Baseline": "#94a3b8"}
best_model = all_data.sort_values("RMSE").index[0]

for ax, metric, asc, label in zip(
    axes,
    ["RMSE", "MAE", "R2"],
    [True,   True,  False],
    ["RMSE (min) — moins c'est mieux", "MAE (min) — moins c'est mieux", "R² — plus c'est mieux"]
):
    vals = all_data[metric].sort_values(ascending=asc)
    bar_colors = [
        "#94a3b8" if i == "Baseline"
        else "#22c55e" if i == best_model
        else "#4f46e5"
        for i in vals.index
    ]
    bars = ax.barh(vals.index, vals.values, color=bar_colors, edgecolor="none", height=0.6)
    ax.axvline(all_data.loc["Baseline", metric], color="#94a3b8", linestyle="--", linewidth=1)
    ax.set_title(label, fontsize=10, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)

legend = [
    mpatches.Patch(color="#22c55e", label="Meilleur modèle"),
    mpatches.Patch(color="#4f46e5", label="Autres modèles"),
    mpatches.Patch(color="#94a3b8", label="Baseline"),
]
fig.legend(handles=legend, loc="lower center", ncol=3, frameon=False, fontsize=10)
plt.suptitle("Comparaison des modèles", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Analyse du meilleur modèle

In [ ]:
# Métriques finales
best_name_tuned = results_df["RMSE"].idxmin()
best_metrics   = all_results[best_name_tuned]

print(f"🏆 Meilleur modèle : {best_name_tuned}")
print(f"   RMSE : {best_metrics['RMSE']:.3f} min  (baseline : {baseline['RMSE']:.3f})")
print(f"   MAE  : {best_metrics['MAE']:.3f} min  (baseline : {baseline['MAE']:.3f})")
print(f"   R²   : {best_metrics['R2']:.4f}       (baseline : {baseline['R2']:.4f})")
print(f"   Réduction RMSE vs baseline : {(1 - best_metrics['RMSE']/baseline['RMSE'])*100:.1f} %")

In [ ]:
# Prédit vs Réel + Résidus
preds = best_pipeline.predict(X_test)
residuals = y_test.to_numpy() - preds

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Prédit vs Réel
axes[0].scatter(y_test, preds, alpha=0.25, s=8, color="#4f46e5", rasterized=True)
lims = [min(y_test.min(), preds.min()), max(y_test.max(), preds.max())]
axes[0].plot(lims, lims, "--", color="#ef4444", linewidth=1.5)
axes[0].set_xlabel("Retard réel (min)")
axes[0].set_ylabel("Retard prédit (min)")
axes[0].set_title("Prédit vs Réel")

# Distribution des résidus
axes[1].hist(residuals, bins=60, color="#4f46e5", alpha=0.8, edgecolor="none")
axes[1].axvline(0, color="#ef4444", linestyle="--")
axes[1].set_xlabel("Résidu (min)")
axes[1].set_ylabel("Fréquence")
axes[1].set_title(f"Résidus — MAE : {mean_absolute_error(y_test, preds):.2f} min")

# Résidus vs Prédit
axes[2].scatter(preds, residuals, alpha=0.2, s=8, color="#4f46e5", rasterized=True)
axes[2].axhline(0, color="#ef4444", linestyle="--")
axes[2].set_xlabel("Retard prédit (min)")
axes[2].set_ylabel("Résidu (min)")
axes[2].set_title("Résidus vs Prédictions")

for ax in axes:
    ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# Importance des variables
final_est = best_pipeline.named_steps["model"]
cat_enc   = best_pipeline.named_steps["prep"].named_transformers_["cat"].named_steps["onehot"]
cat_names = cat_enc.get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_names = NUMERIC_FEATURES + cat_names

imp_attr = getattr(final_est, "feature_importances_", getattr(final_est, "coef_", None))
if imp_attr is not None:
    importances = pd.Series(np.abs(imp_attr), index=all_names).sort_values(ascending=False)
    top = importances.head(20).sort_values()

    fig, ax = plt.subplots(figsize=(10, 7))
    bars = ax.barh(top.index, top.values, color="#4f46e5", edgecolor="none", height=0.7)
    ax.set_xlabel("Importance")
    ax.set_title("Top 20 — Importance des variables", fontsize=12, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

    print("Top 10 variables :")
    for feat, val in importances.head(10).items():
        bar = "█" * int(val / importances.max() * 30)
        print(f"  {feat:50s} {bar} {val:.5f}")
else:
    print("Importance non disponible pour ce type de modèle.")

## 7. Vérification finale

Test de la fonction `predict_delay` telle qu'elle sera appelée dans le dashboard.

In [ ]:
import warnings as _warnings
from scripts.model.predict import predict_delay

loaded = joblib.load("model.joblib")
loaded_pipeline = loaded["pipeline"] if isinstance(loaded, dict) else loaded
route_stats = pd.read_csv("route_stats.csv")

print(f"Modèle chargé : {loaded.get('model_name', '?') if isinstance(loaded, dict) else type(loaded).__name__}")
print(f"Routes disponibles : {len(route_stats):,}\n")

examples = [
    ("PARIS MONTPARNASSE", "BORDEAUX SAINT JEAN", pd.Timestamp("2025-01-15")),
    ("PARIS MONTPARNASSE", "NANTES",              pd.Timestamp("2025-07-10")),
    ("PARIS LYON",         "MARSEILLE ST CHARLES",pd.Timestamp("2025-03-20")),
    ("LILLE EUROPE",       "PARIS NORD",          pd.Timestamp("2025-12-01")),
]

saison_map = {1:"hiver",2:"hiver",3:"printemps",4:"printemps",5:"printemps",
              6:"été",7:"été",8:"été",9:"automne",10:"automne",11:"automne",12:"hiver"}

for dep, arr, date in examples:
    try:
        with _warnings.catch_warnings(record=True) as caught:
            _warnings.simplefilter("always")
            pred = predict_delay(dep, arr, date, loaded_pipeline, route_stats)
        saison = saison_map[date.month]
        note = f"  [~estimation: {caught[0].message}]" if caught else ""
        print(f"  {dep:28s} → {arr:28s}  [{date.strftime('%d/%m/%Y')} / {saison}]  →  {pred:.1f} min{note}")
    except ValueError as e:
        print(f"  ⚠ {e}")

In [ ]:
# Résumé des fichiers générés
import os, glob

print("Fichiers générés :")
for f in ["model.joblib", "route_stats.csv"] + sorted(glob.glob("models/*.joblib")) + ["models/metadata.json"]:
    if os.path.exists(f):
        size = os.path.getsize(f)
        print(f"  ✓ {f:40s} {size/1024:.0f} Ko")
    else:
        print(f"  ✗ {f} (manquant)")